# Octo в LIBERO (глава 3, шаг 1)

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/chapter3/octo/notebooks/libero_octo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Те же 10 задач LIBERO-Spatial, те же начальные расстановки (0–2) и тот же лимит 220 шагов, что
у OpenVLA (глава 1, 76,7%) и OpenVLA-OFT (глава 2, 100%).

Модель — **Octo-Small 1.5** (27 млн параметров), дообученная на демонстрациях LIBERO:
`cyrusneary/octo-finetuned-libero` (шаг 60000). Это не модель авторов Octo: её выложил сторонний
исследователь, обучена сразу на всех четырёх наборах LIBERO. Входы — **внешняя камера и команда**,
как у OpenVLA в главе 1; камеры на запястье и датчиков нет. Для сравнения: авторы OpenVLA дообучили
Octo на LIBERO-Spatial сами и получили **78,9%** (Табл. 12 их статьи).

Это шаг 1: убедиться, что Octo скачивается и работает в LIBERO. Шаг 2 — дообучить Octo самому
с разными входами (камера, камера на запястье, датчики) — отдельный ноутбук.

Видеокарта: хватает бесплатной **T4**.

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Установка (~10 мин)

Отдельное окружение Python 3.10: Octo требует JAX 0.4.20, LIBERO — robosuite 1.4.1.

In [ ]:
BRANCH = "chapter3/octo"   # после слияния в main поменять на "main"
PY = "/content/venv-octo/bin/python"
REPO_DIR = "/content/vla-manipulation-research"
JAX_LINKS = "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!cd vla-manipulation-research && git fetch -q origin && git reset -q --hard origin/{BRANCH} && git log --oneline -1
!test -d LIBERO || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
!sed -i 's/init_states = torch.load(init_states_path)/init_states = torch.load(init_states_path, weights_only=False)/' LIBERO/libero/libero/benchmark/__init__.py
!test -d octo || git clone -q --depth 1 https://github.com/octo-models/octo.git

!pip install -q uv
!test -d /content/venv-octo || uv venv -q --python 3.10 /content/venv-octo
!uv pip install -q --python {PY} -r {REPO_DIR}/notebooks/requirements-libero-octo.txt
!uv pip install -q --python {PY} --no-deps -e /content/octo "bddl==1.0.1"
# PyTorch нужен LIBERO только для чтения начальных расстановок — хватает версии без видеокарты
!uv pip install -q --python {PY} --index-url https://download.pytorch.org/whl/cpu "torch==2.2.0"
# JAX с поддержкой CUDA 12 (cuDNN 8.9 — под эту сборку jaxlib)
!uv pip install -q --python {PY} -f {JAX_LINKS} "jax[cuda12_pip]==0.4.20" "nvidia-cudnn-cu12==8.9.7.29"
!{PY} -c "import sys, jax, flax, tensorflow as tf, robosuite, octo; print('Python', sys.version.split()[0], '| jax', jax.__version__, jax.devices(), '| flax', flax.__version__, '| tf', tf.__version__)"

## 3. Первый эпизод (~3–5 мин, из них часть — скачивание модели)

Задача 0, расстановка 0. Видео: слева — внешняя камера (её получает модель), справа — камера на
запястье (только для нас).

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

SUITE = "libero_spatial"
SCRIPT = f"{REPO_DIR}/scripts/libero_octo_eval.py"
ARGS = f"--libero-root /content/LIBERO --suite {SUITE}"
OUT = f"/content/results/octo_{SUITE}"

!{PY} {SCRIPT} {ARGS} --tasks 0 --trials 1 --out {OUT}/first

from IPython.display import Video, display
display(Video(f"{OUT}/first/{SUITE}_task00_trial00.mp4", embed=True, width=512))

## 4. Серия: 10 задач × 3 попытки

Те же 30 эпизодов, что в главах 1 и 2.

In [ ]:
TASKS = " ".join(str(t) for t in range(10))
!{PY} {SCRIPT} {ARGS} --tasks {TASKS} --trials 3 --out {OUT}/series

## 5. Результаты

Для сравнения на тех же 30 эпизодах: OpenVLA (глава 1) — **23/30 (76,7%)**, ~103 с на успешную попытку;
OpenVLA-OFT (глава 2) — **30/30**, ~18 с.

In [ ]:
import json
import numpy as np
import pandas as pd

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * max(c - h, 0), 100 * min(c + h, 1)

df = pd.read_csv(f"{OUT}/series/summary.csv")
meta = json.load(open(f"{OUT}/series/meta.json"))
k, n = int(df.success.sum()), len(df)
lo, hi = wilson(k, n)
print(f"Octo: {k}/{n} ({100 * k / n:.1f}%), 95% ДИ [{lo:.0f}%, {hi:.0f}%]")
print(f"запрос к модели {meta['mean_infer_s_per_query']:.3f} с, пачка {meta['action_horizon']} действий")
print(f"успешная попытка в среднем {df[df.success].wall_s.mean():.0f} с")
df.groupby("task_id").agg(успех=("success", "sum"), попыток=("success", "size"), шагов=("steps", "mean"))

## 6. Сохранение результатов

Таблицы по шагам (`*.csv`), сводка и видео — в архив.

In [ ]:
!cd /content/results && zip -qr /content/results_octo.zip octo_{SUITE} && ls -lh /content/results_octo.zip
from google.colab import files
files.download("/content/results_octo.zip")